# Task 1.2 — Key Assumptions (8 marks)

**Paper**: *Robust Point Set Registration Using Gaussian Mixture Models* — Bing Jian and Baba C. Vemuri, IEEE TPAMI, 2011.

---

## Assumption 1: Isotropic Gaussians with a Shared Bandwidth Parameter

**Assumption**: Every point in both the model and scene sets is represented by the same isotropic Gaussian kernel $\phi(x | \mu_i, \sigma^2 I)$ with a **single, shared bandwidth** $\sigma$. No point receives a different variance or a full covariance matrix.

**Why the method needs it**: The closed-form expression for the L2 distance between two Gaussian mixtures (Eq. 4) relies on all components having the same spherical covariance $\sigma^2 I$. If each point had a different covariance, the summations in the Gauss transform would become heterogeneous and the efficient pairwise evaluation in the inner loops would no longer factor cleanly. Additionally, the multi-scale annealing strategy (Algorithm 1) works by controlling a single global $\sigma$ — it would be unclear how to anneal point-specific bandwidths.

**Violation scenario**: In a LiDAR scan of a building, points on a flat wall have very low positional uncertainty while points on edges or corners have high uncertainty due to mixed returns. A single $\sigma$ cannot capture this heterogeneous noise: too large wastes information on the wall, too small amplifies noise on edges.

**Paper reference**: Section 2, Equation 2, and the discussion immediately following Equation 2 where the authors state *"we simplify the model by assuming that all the component densities share the same covariance matrix $\sigma^2 I$"*.

---

## Assumption 2: Similar Sampling Density Between Point Sets

**Assumption**: The density-based registration framework assumes that the model and scene point sets have **roughly similar sampling rates** across the shape. The Gaussian mixture representation treats every point equally (uniform weights $w_i = 1/k$), so regions with more points automatically contribute more to the density.

**Why the method needs it**: The L2 distance $\int (f(x) - g(x))^2 dx$ compares two density functions. If one point set is densely sampled in one region and the other is sparse there, the density peaks will have different heights even if they represent the same underlying shape. The optimiser will then try to match density magnitudes rather than shape geometry, leading to biased registration. The authors explicitly note this: *"the general idea of this density based registration approach favors the pair of point sets of the similar sampling rates"* (Section 2).

**Violation scenario**: Registering a 3D face scan from a structured-light sensor (dense, uniform sampling) against a photogrammetry reconstruction (sparse on smooth surfaces, dense on textured areas). The density mismatch would cause the registration to be pulled toward high-density textured regions rather than accurately aligning the overall shape.

**Paper reference**: Section 2, the paragraph beginning *"Note that the general idea of this density based registration approach favors the pair of point sets of the similar sampling rates."*

---

## Assumption 3: The Underlying Deformation Field is Smooth

**Assumption**: The non-rigid transformation between the model and scene is assumed to be a **smooth, continuous** deformation field. In the paper's implementation, this is enforced via thin-plate spline (TPS) regularisation, which penalises the bending energy $\text{trace}(W^T K W)$ — effectively penalising large second-order spatial derivatives of the deformation.

**Why the method needs it**: The TPS regularisation term $\lambda \cdot \text{trace}(W^T K W)$ in the objective function (Eq. 14) acts as a prior on the smoothness of the transformation. Without this smoothness assumption, the optimisation is severely ill-posed for non-rigid registration: with enough degrees of freedom, any model point can be mapped to any scene point. The regularisation constrains the solution space to physically plausible deformations.

**Violation scenario**: Registering two frames of a video sequence where an object undergoes a **fracture or tearing** — for example, a clay model being torn into two pieces. The deformation field has a discontinuity at the tear boundary which TPS cannot represent. The registration would produce a smoothed-out compromise that fails to capture the actual deformation on either side of the discontinuity.

**Paper reference**: Section 5, Equation 14, where the TPS bending energy is added as the regularisation term.